In [1]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from xgboost.callback import EarlyStopping




This line loads the dataset from a CSV file into a pandas DataFrame called df.
From this point on, df becomes the main data structure used for exploration, cleaning, transformation, and model training. Each row represents an individual’s financial profile along with the investment products recommended for that profile, and each column corresponds to a specific attribute (such as age, risk tolerance, or investment recommendations).

In [2]:
df = pd.read_csv("investment_recommendations_10000.csv")

This cell checks the size of the dataset by retrieving the number of rows and columns in df.
The result is a tuple in the form (rows, columns), which tells how many individual records and how many features describe each record.

In [3]:
data_shape = df.shape
data_shape

(10000, 6)

This cell prints a concise technical summary of the dataset.
It shows each column’s name, data type, number of non-null values, and the overall memory usage of the DataFrame.

The goal here is to quickly spot potential problems such as missing values, incorrect data types (for example, numbers stored as strings), or columns that may need preprocessing before modeling.

In [4]:
data_info = df.info()
data_info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Individual Goals                 10000 non-null  object
 1   Age                              10000 non-null  int64 
 2   Gender                           10000 non-null  object
 3   Risk Tolerance                   10000 non-null  object
 4   Financial Literacy               10000 non-null  int64 
 5   Recommended Investment Products  10000 non-null  object
dtypes: int64(2), object(4)
memory usage: 468.9+ KB


This cell generates descriptive statistics for all numerical columns in the dataset.
It reports key summary measures such as count, mean, standard deviation, minimum, maximum, and quartiles.

This helps to understand the distribution and scale of numeric features (like age or financial literacy), identify outliers, and decide whether any normalization or data cleaning might be necessary later.

In [5]:
data_description= df.describe()
data_description

,Age,Financial Literacy
count,10000.00000,10000.000000
mean,39.84960,3.025500
std,23.27352,1.413807
min,0.00000,1.000000
25%,20.00000,2.000000
50%,40.00000,3.000000
75%,60.00000,4.000000
max,80.00000,5.000000


This cell checks whether the dataset contains any missing values at all.
It returns a single Boolean value: True if at least one cell is null, and False if the dataset is completely filled.

This is a blunt but useful check if it comes back True, you know upfront that you’ll need to handle missing data before training a model.

In [6]:
has_nulls = df.isnull().values.any()
has_nulls

False

This cell retrieves the names of all columns in the dataset.
Seeing the column list helps you understand what features are available, verify that expected fields are present, and plan which columns will be used as inputs, targets, or dropped during preprocessing.

In [7]:
column_names = df.columns
column_names

Index(['Individual Goals', 'Age', 'Gender', 'Risk Tolerance',
       'Financial Literacy', 'Recommended Investment Products'],
      dtype='object')

This cell displays the first five rows of the dataset.

In [8]:
head = df.head()
head

,Individual Goals,Age,Gender,Risk Tolerance,Financial Literacy,Recommended Investment Products
0,saving for retirement,44,Male,Medium,2,"Mutual Funds, PPF, NPS, Stocks, Real Estate"
1,buying a house,47,Male,Medium,2,"Real Estate Investment Trusts (REITs), Home Lo..."
2,saving for education,64,Male,Low,4,"National Scholarship Scheme, Education Bonds, ..."
3,saving for retirement,67,Female,High,2,"Mutual Funds, PPF, NPS, Stocks, Real Estate"
4,saving for education,67,Male,High,3,"National Scholarship Scheme, Education Bonds, ..."


This block measures how diverse the values are in each column.
For every column, it counts how many unique values appear and stores that count in the `unique_element_count` dictionary.

The purpose is to quickly identify which columns are categorical with limited categories, which ones are high-cardinality, and which might need encoding or special handling. The final output shows the number of distinct values per column, helping guide later preprocessing and feature selection.


In [9]:
unique_element_count={}
unique_element={}
for column in column_names:
    count = df[column].nunique()
    unique_element_count[column] = (count)
    unique_elements = (count, df[column].unique())
unique_element_count, unique_elements

({'Individual Goals': 6,
  'Age': 81,
  'Gender': 2,
  'Risk Tolerance': 3,
  'Financial Literacy': 5,
  'Recommended Investment Products': 18},
 (18,
  array(['Mutual Funds, PPF, NPS, Stocks, Real Estate',
         'Real Estate Investment Trusts (REITs), Home Loans, Stocks, Real Estate',
         'National Scholarship Scheme, Education Bonds, Education Loans, Stocks, Mutual Funds',
         "Sukanya Samriddhi Yojana, Children's Mutual Funds, Real Estate Investment Trusts (REITs), Home Loans, Stocks",
         "Children's Mutual Funds, Personal Loans, Consumer Stocks, Gold, Mutual Funds",
         'Personal Loans, Consumer Stocks, Gold, Mutual Funds, Cryptocurrency',
         'Stand-Up India Scheme, Equity Shares, Business Loans, Stocks, Real Estate',
         "Children's Mutual Funds, National Scholarship Scheme, Education Bonds, Education Loans, Stocks",
         'Travel Insurance, Short-Term Mutual Funds, Savings Account, Stocks, Cryptocurrency',
         "Sukanya Samriddhi Yojana, 

This cell extracts all unique investment products mentioned in the dataset.
Each row may contain multiple recommended products stored as a comma-separated string, so the code splits each row and collects every individual product into a set.

Using a set automatically removes duplicates, leaving you with the full universe of distinct investment options that the model may later learn to recommend.

In [10]:
investment_options=set()
for row in df['Recommended Investment Products']:
    investment=row.split(",")
    for i in range(len(investment)):
        investment_options.add(investment[i])
    
investment_options

{' Business Loans',
 " Children's Mutual Funds",
 ' Consumer Stocks',
 ' Cryptocurrency',
 ' Education Bonds',
 ' Education Loans',
 ' Equity Shares',
 ' Gold',
 ' Home Loans',
 ' Mutual Funds',
 ' NPS',
 ' National Scholarship Scheme',
 ' PPF',
 ' Personal Loans',
 ' Real Estate',
 ' Real Estate Investment Trusts (REITs)',
 ' Savings Account',
 ' Short-Term Mutual Funds',
 ' Stand-Up India Scheme',
 ' Stocks',
 ' Travel Insurance',
 "Children's Mutual Funds",
 'Mutual Funds',
 'National Scholarship Scheme',
 'Personal Loans',
 'Real Estate Investment Trusts (REITs)',
 'Stand-Up India Scheme',
 'Sukanya Samriddhi Yojana',
 'Travel Insurance'}

This cell prints each row’s recommended investment products as a list by splitting the comma-separated string.

The purpose is to visually verify how the products are stored, check for formatting issues (like extra spaces or inconsistent separators), and ensure that the splitting logic will correctly separate multiple products for further processing.

In [11]:

for row in df['Recommended Investment Products']:
    print(row.split(","))
    

['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
['Real Estate Investment Trusts (REITs)', ' Home Loans', ' Stocks', ' Real Estate']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['Sukanya Samriddhi Yojana', " Children's Mutual Funds", ' Real Estate Investment Trusts (REITs)', ' Home Loans', ' Stocks']
['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
["Children's Mutual Funds", ' Personal Loans', ' Consumer Stocks', ' Gold', ' Mutual Funds']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['National Scholarship Scheme', ' Education Bonds', ' Educ



This cell transforms the dataset from a **multi-product format** into a **long format** where each row contains only **one recommended product** per person.

* `str.split(', ')` separates the products into lists.
* `.explode()` creates a separate row for each product, duplicating all other columns so the person’s profile is preserved.
* `.to_frame()` converts the series back to a DataFrame, and `.join()` reattaches the other columns.
* `.reset_index(drop=True)` cleans up the row indexing.

The result, `df_long`, is easier to work with for modeling, because each row now represents a single “person → product” pair.


In [12]:
# Split the products and explode → one product per row, keeping all other columns duplicated
df_long = (
    df['Recommended Investment Products']
    .str.split(', ')
    .explode()
    .explode()
    .to_frame('Recommended Investment Product')
    .join(df.drop('Recommended Investment Products', axis=1))
    .reset_index(drop=True)
)

df_long

,Recommended Investment Product,Individual Goals,Age,Gender,Risk Tolerance,Financial Literacy
0,Mutual Funds,saving for retirement,44,Male,Medium,2
1,PPF,saving for retirement,44,Male,Medium,2
2,NPS,saving for retirement,44,Male,Medium,2
3,Stocks,saving for retirement,44,Male,Medium,2
4,Real Estate,saving for retirement,44,Male,Medium,2
...,...,...,...,...,...,...
49063,Personal Loans,purchasing for self,71,Male,Medium,1
49064,Consumer Stocks,purchasing for self,71,Male,Medium,1
49065,Gold,purchasing for self,71,Male,Medium,1
49066,Mutual Funds,purchasing for self,71,Male,Medium,1


In [13]:
df_long.shape

(49068, 6)

This cell does two things:

all_products stores a list of all unique investment products in the long-format dataset. This is useful later for generating predictions and recommendations.

person_cols defines the list of columns that describe an individual’s profile (their goals, age, gender, risk tolerance, and financial literacy). These are the features that the model will use to predict suitable investment products.

Essentially, it separates what we’re predicting (products) from who we’re predicting for (person attributes).

In [14]:
all_products = df_long['Recommended Investment Product'].unique()
person_cols = ['Individual Goals', 'Age', 'Gender', 'Risk Tolerance', 'Financial Literacy']

df = df_long.copy() creates a separate working copy of the long-format data so the original df_long remains unchanged.

In [15]:
df = df_long.copy()

# Create binary target (1 = this product was recommended for this person)
# df['rec'] = 1

This cell converts all categorical string columns into numeric codes, which machine learning models like XGBoost require.

categorical_columns lists the features and target that need encoding.

For each column, a LabelEncoder is created and used to map each unique string to a unique integer.

The encoded values are stored in new columns (e.g., Gender_encoded).

The encoders themselves are saved in label_encoders so the same mapping can be applied later during prediction or for new users.

The print statement confirms how many unique values each column has, which helps track data consistency.

In [16]:
# Label-encode ALL categorical columns (strings → integers)
categorical_columns = ['Individual Goals', 'Gender', 'Risk Tolerance', 'Recommended Investment Product']

label_encoders = {}   # Save them so we can reuse during inference!

for col in categorical_columns:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le                     # ← keep for later use
    print(f"{col:30} → {len(le.classes_)} unique values")



Individual Goals               → 6 unique values
Gender                         → 2 unique values
Risk Tolerance                 → 3 unique values
Recommended Investment Product → 22 unique values


In [17]:
df

,Recommended Investment Product,Individual Goals,Age,Gender,Risk Tolerance,Financial Literacy,Individual Goals_encoded,Gender_encoded,Risk Tolerance_encoded,Recommended Investment Product_encoded
0,Mutual Funds,saving for retirement,44,Male,Medium,2,4,1,2,9
1,PPF,saving for retirement,44,Male,Medium,2,4,1,2,12
2,NPS,saving for retirement,44,Male,Medium,2,4,1,2,10
3,Stocks,saving for retirement,44,Male,Medium,2,4,1,2,19
4,Real Estate,saving for retirement,44,Male,Medium,2,4,1,2,14
...,...,...,...,...,...,...,...,...,...,...
49063,Personal Loans,purchasing for self,71,Male,Medium,1,2,1,2,13
49064,Consumer Stocks,purchasing for self,71,Male,Medium,1,2,1,2,2
49065,Gold,purchasing for self,71,Male,Medium,1,2,1,2,7
49066,Mutual Funds,purchasing for self,71,Male,Medium,1,2,1,2,9


This cell prepares the data for training the machine learning model:

1. `feature_cols` lists the columns used as input features for prediction (encoded categorical features plus numeric ones like age and financial literacy).
2. `X` is the feature matrix, and `y` is the target vector (the encoded recommended investment product).
3. `train_test_split` divides the data into training and validation sets:

   * 80% for training (`X_train`, `y_train`)
   * 20% for validation (`X_val`, `y_val`)
   * `stratify=y` ensures that the distribution of investment products is preserved in both sets.
   * `random_state=42` guarantees reproducibility.

This step ensures the model is trained on one set of data and evaluated on a separate, unbiased set to monitor performance.


In [18]:
feature_cols = [
    'Individual Goals_encoded',
    'Gender_encoded', 
    'Risk Tolerance_encoded',
    'Age',
    'Financial Literacy'
]

X = df[feature_cols]
y = df['Recommended Investment Product_encoded']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

This cell trains an **XGBoost classifier** to predict the recommended investment product for each user profile:

1. `EarlyStopping` is set to monitor the validation **AUC** and stop training if it doesn’t improve for 50 rounds, preventing overfitting and saving the best model.
2. The `XGBClassifier` is configured with:

   * `n_estimators=2000` (maximum number of trees)
   * `max_depth=8` (tree complexity)
   * `learning_rate=0.05` (step size for boosting)
   * `subsample` and `colsample_bytree` for randomness and regularization
   * `tree_method='hist'` for fast training on large datasets
   * `callbacks=[early_stop]` to apply early stopping
3. `model.fit()` trains the model on the training set and evaluates it on the validation set every 50 rounds (`verbose=50`).
4. After training, it prints:

   * The best iteration (number of trees used)
   * The best validation AUC score

In short, this cell builds the predictive model that will later power the personalized investment recommendations.


In [19]:
early_stop = EarlyStopping(rounds=50, metric_name='auc', save_best=True)

model = xgb.XGBClassifier(
    n_estimators=2000,           
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    tree_method='hist',
    n_jobs=-1,
    verbosity=1,
    callbacks=[early_stop]      
)
\
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],  
    verbose=50
)

print("Training finished!")
print("Best iteration :", model.get_booster().best_iteration)
print("Best AUC        :", model.get_booster().best_score)

[0]	validation_0-auc:0.78383
[50]	validation_0-auc:0.77902
[52]	validation_0-auc:0.77769
Training finished!
Best iteration : 3
Best AUC        : 0.8290575817999358


This cell defines a **function to generate personalized top-5 investment recommendations** for a single user profile:

1. It first **encodes the user’s input** (age, gender, risk tolerance, financial literacy, and goals) using the same `LabelEncoder`s used during training, ensuring consistency with the model.
2. It builds a candidate dataset containing **all possible products** for that user, with the user’s features repeated for each product.
3. `model.predict_proba` estimates the likelihood that each product is suitable for the user.
4. The products are **ranked by predicted probability**, and the top 5 are returned in a DataFrame along with their predicted probabilities.

This function allows you to quickly get the five most appropriate investments for any new individual profile without retraining the model.


In [20]:
# TOP-5 RECOMMENDATION FUNCTION 
def recommend_top5(age, gender, risk_tolerance, financial_literacy, individual_goals):
    
    # Encode the input using the SAME label encoders we saved during training
    goals_enc     = label_encoders['Individual Goals'].transform([individual_goals])[0]
    gender_enc    = label_encoders['Gender'].transform([gender])[0]
    risk_enc      = label_encoders['Risk Tolerance'].transform([risk_tolerance])[0]
    
    # All possible products
    all_products = label_encoders['Recommended Investment Product'].classes_
    
    # Build candidate rows (one per product)
    candidates = []
    for prod in all_products:
        prod_enc = label_encoders['Recommended Investment Product'].transform([prod])[0]
        candidates.append([
            goals_enc,
            gender_enc,
            risk_enc,
            age,
            financial_literacy
        ])
    
    candidates_df = pd.DataFrame(
        candidates,
        columns=feature_cols,
        index=all_products
    )
    
    # Predict probability
    probs = model.predict_proba(candidates_df)[:, 1]
    
    # Rank and return top 5
    
    # result = pd.DataFrame({
    #     'Recommended Investment Product': all_products,
    # }).head(5)
    
    result = pd.DataFrame({
        'Recommended Investment Product': all_products,
        'probability': probs
    }).sort_values('probability', ascending=False).head(5).reset_index(drop=True)
    
    return result



In [23]:
import pandas as pd
import requests
import json

# Assume this function exists and returns a DataFrame
top5 = recommend_top5(
    age=35,
    gender='Male',
    risk_tolerance='High',
    financial_literacy=4,
    individual_goals='saving for retirement'
)

products = top5['Recommended Investment Product'].tolist()
probs = (top5['probability'] * 100).round(1).astype(str) + "%"

# prompt
prompt = f"""
You are a warm, trusted financial advisor speaking to a 35-year-old man who wants to save for retirement.
He is comfortable with high risk and has good financial knowledge.

Here are the top 5 investments I recommend for him:

1. {products[0]} — best match ({probs[0]} confidence)
2. {products[1]} — very strong ({probs[1]})
3. {products[2]} — excellent fit ({probs[2]})
4. {products[3]} — solid choice ({probs[3]})
5. {products[4]} — good addition ({probs[4]})

Please explain each one in 2–3 simple, encouraging sentences using "you".
Start with: "Great news! Based on your profile, here are the 5 best investments for your retirement goal:"
End with: "You're in a fantastic position — let's get started whenever you're ready!"
"""

def ask_ollama(prompt, model="qwen2.5:3b-instruct"):
    try:
        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
                "options": {
                    "temperature": 0.7,
                    "num_ctx": 8192  # helps longer coherent outputs
                }
            },
            timeout=120  # 2 minutes — generation can take time on CPU
        )
        response.raise_for_status()  # raises if not 200

        data = response.json()
        return data['message']['content']

    except requests.exceptions.RequestException as e:
        return f"Connection/API error: {str(e)}\n→ Is Ollama running? (try: ollama serve)\n→ Test: curl http://localhost:11434"
    except KeyError:
        return "Unexpected Ollama response format — check model is loaded."
    except Exception as e:
        return f"Unexpected error: {str(e)}"

print("\n" + "═"*70)
print("YOUR PERSONAL FINANCIAL ADVISOR Response")
print("═"*70)

explanation = ask_ollama(prompt, model="qwen2.5:3b-instruct")
print(explanation)


══════════════════════════════════════════════════════════════════════
YOUR PERSONAL FINANCIAL ADVISOR 
══════════════════════════════════════════════════════════════════════
Great news! Based on your profile, here are the 5 best investments for your retirement goal:

1. For those comfortable with high risk and good financial knowledge like yourself, Business Loans offer a solid return of about 4.2%, which can help grow your wealth.
2. Children's Mutual Funds are very strong in offering growth potential suitable for long-term investment. They're perfect for diversifying your portfolio to include some equity exposure.
3. The Sukanya Samriddhi Yojana is an excellent fit for you as a 35-year-old professional with good financial knowledge, giving you access to high-interest rates on savings without tax implications.
4. Stocks provide solid returns and are a natural extension of your comfort level with risk. You're in the right place when it comes to investments that offer potential growth